# 14 — Account Ledger (Amazon FAR style)

Money transfers between accounts — correctness and atomicity under concurrency. Parts 1-3 build the
core (concurrency at Part 3, where **deadlock avoidance** is the lesson); Parts 4-5 are stretch tasks
(idempotent atomic batches, then sharding + parallel replay). Fill stubs, run each test cell, peek at
the collapsed `ref_` solutions only after trying. Amounts are positive integers.

---

## Part 1 — Single-threaded ledger

`Ledger` with `open(acct, balance=0)`, `balance(acct)`, `deposit(acct, amt)`, `withdraw(acct, amt)`,
and `transfer(src, dst, amt)`. Withdrawals/transfers that would overdraft raise `ValueError`, and a
failed transfer must leave **both** balances unchanged (atomic).

**Lock down:** check funds before moving money; a transfer is debit+credit as one unit (no partial
state on failure).

In [ ]:
class Ledger:
    def __init__(self):
        raise NotImplementedError

    def open(self, acct, balance=0):
        raise NotImplementedError

    def balance(self, acct):
        raise NotImplementedError

    def deposit(self, acct, amt):
        raise NotImplementedError

    def withdraw(self, acct, amt):
        raise NotImplementedError

    def transfer(self, src, dst, amt):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    L = Ledger(); L.open("a", 100); L.open("b", 0)
    L.transfer("a", "b", 30)
    assert L.balance("a") == 70 and L.balance("b") == 30
    try:
        L.transfer("a", "b", 1000)
    except ValueError:
        pass
    else:
        raise AssertionError("expected overdraft ValueError")
    assert L.balance("a") == 70 and L.balance("b") == 30      # unchanged after failed transfer
    L.deposit("b", 5); assert L.balance("b") == 35
    print("Part 1 OK")

_t1()

## Part 2 — Replay a transfer log

`replay_txns(initial, txns) -> dict`: a pure function applying transfers (`(src, dst, amt)`) in order
to a starting balance map, raising `ValueError` on the first overdraft. Total money is **conserved**
across transfers.

**Lock down:** transfers conserve the sum; order matters (a later transfer may depend on an earlier
one); don't mutate `initial`.

In [ ]:
def replay_txns(initial, txns):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    init = {"a": 100, "b": 50, "c": 0}
    txns = [("a", "b", 30), ("b", "c", 20), ("a", "c", 10)]
    final = replay_txns(init, txns)
    assert final == {"a": 60, "b": 60, "c": 30}
    assert sum(final.values()) == sum(init.values())          # conserved
    assert init == {"a": 100, "b": 50, "c": 0}                # input untouched
    try:
        replay_txns({"a": 5, "b": 0}, [("a", "b", 10)])
    except ValueError:
        pass
    else:
        raise AssertionError("expected overdraft ValueError")
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent transfers (deadlock-free)

`ConcurrentLedger`: `open`, `balance`, `transfer`, and `total()`, all thread-safe with a **lock per
account**. Many threads transfer between random accounts at once. Two transfers `A→B` and `B→A` that
grab their two locks in argument order can deadlock — so always acquire the two account locks in a
**fixed global order** (e.g. sorted account id).

**The invariant:** under heavy concurrency, total money is **conserved** and no balance goes negative.
`total()` takes a consistent snapshot (acquire all locks in the same global order). **Discuss:** lock
ordering for deadlock freedom, lock granularity, and why a single global lock is correct but kills
throughput.

In [ ]:
import threading


class ConcurrentLedger:
    def __init__(self):
        raise NotImplementedError

    def open(self, acct, balance=0):
        raise NotImplementedError

    def balance(self, acct):
        raise NotImplementedError

    def transfer(self, src, dst, amt):
        raise NotImplementedError

    def total(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    import random
    L = ConcurrentLedger()
    K = 10
    for i in range(K):
        L.open(i, 1000)
    start_total = K * 1000

    def worker(seed):
        rng = random.Random(seed)
        for _ in range(500):
            a, b = rng.randrange(K), rng.randrange(K)
            if a == b:
                continue
            try:
                L.transfer(a, b, rng.randrange(1, 50))
            except ValueError:
                pass

    ts = [threading.Thread(target=worker, args=(s,)) for s in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert L.total() == start_total                  # money conserved despite races
    assert all(L.balance(i) >= 0 for i in range(K))  # never went negative
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Idempotent, atomic batches

`apply_batch(initial, txns) -> dict`: `txns` are `(txn_id, src, dst, amt)`.
- **Idempotent:** a repeated `txn_id` is applied at most once (safe retries).
- **Atomic:** if any transfer in the batch would overdraft, raise `ValueError` and leave `initial`
  completely unchanged (all-or-nothing).

**Discuss:** dedup keys for exactly-once semantics, two-phase apply (validate then commit), and why
working on a copy gives you free rollback.

In [ ]:
def apply_batch(initial, txns):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    init = {"a": 100, "b": 0}
    txns = [("t1", "a", "b", 30), ("t1", "a", "b", 30), ("t2", "a", "b", 20)]
    final = apply_batch(init, txns)
    assert final == {"a": 50, "b": 50}               # t1 applied once, t2 once
    assert init == {"a": 100, "b": 0}                # input untouched
    bad = [("t1", "a", "b", 60), ("t2", "a", "b", 60)]   # second would overdraft
    try:
        apply_batch(init, bad)
    except ValueError:
        pass
    else:
        raise AssertionError("expected atomic batch to fail")
    assert init == {"a": 100, "b": 0}                # rolled back (never touched)
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded ledger + parallel replay

**(a)** `ShardedLedger(n_shards)` partitions accounts across shards by `hash(acct) % n_shards` (use
**int** account ids so placement is deterministic). `open`/`balance`/`transfer`; a transfer whose two
accounts fall in different shards raises `ValueError` (cross-shard needs a distributed transaction).

**(b)** `parallel_replay(shards, num_procs) -> list[dict]`: replay each shard's `(initial, txns)` log
to final balances in parallel with `ProcessPoolExecutor`; worker is `ledger_workers.replay_shard`
(overdrafting transfers are skipped).

**Discuss:** why same-shard transfers are a single lock while cross-shard needs 2PC/sagas; sharding by
account to keep transfers local.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import ledger_workers


class ShardedLedger:
    def __init__(self, n_shards):
        raise NotImplementedError

    def open(self, acct, balance=0):
        raise NotImplementedError

    def balance(self, acct):
        raise NotImplementedError

    def transfer(self, src, dst, amt):
        raise NotImplementedError


def parallel_replay(shards, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    L = ShardedLedger(4)
    for acct in (0, 1, 4, 5):
        L.open(acct, 100)
    L.transfer(0, 4, 30)                              # 0 and 4 share shard 0
    assert L.balance(0) == 70 and L.balance(4) == 130
    try:
        L.transfer(0, 1, 10)                          # different shards
    except ValueError:
        pass
    else:
        raise AssertionError("expected cross-shard ValueError")
    shards = [({0: 100, 4: 0}, [(0, 4, 40), (0, 4, 80)]), ({1: 50, 5: 0}, [(1, 5, 20)])]
    assert parallel_replay(shards, 2) == [{0: 60, 4: 40}, {1: 30, 5: 20}]
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Cross-shard transfers: two-phase commit / sagas, idempotency keys, compensation.
- Double-entry bookkeeping & an append-only journal; reconciliation.
- Optimistic concurrency (version/CAS) vs pessimistic locking.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from contextlib import ExitStack
from concurrent.futures import ProcessPoolExecutor
import ledger_workers


class RefLedger:
    def __init__(self):
        self._b = {}

    def open(self, acct, balance=0):
        self._b[acct] = balance

    def balance(self, acct):
        return self._b[acct]

    def deposit(self, acct, amt):
        self._b[acct] += amt

    def withdraw(self, acct, amt):
        if self._b[acct] < amt:
            raise ValueError("insufficient funds")
        self._b[acct] -= amt

    def transfer(self, src, dst, amt):
        if self._b[src] < amt:
            raise ValueError("insufficient funds")    # check before any mutation -> atomic
        self._b[src] -= amt
        self._b[dst] += amt


def ref_replay_txns(initial, txns):
    bal = dict(initial)
    for src, dst, amt in txns:
        if bal[src] < amt:
            raise ValueError("overdraft")
        bal[src] -= amt
        bal[dst] += amt
    return bal


class RefConcurrentLedger:
    def __init__(self):
        self._b = {}
        self._locks = {}
        self._glock = threading.Lock()

    def open(self, acct, balance=0):
        with self._glock:
            self._b[acct] = balance
            self._locks[acct] = threading.Lock()

    def balance(self, acct):
        with self._locks[acct]:
            return self._b[acct]

    def transfer(self, src, dst, amt):
        if src == dst:
            return
        first, second = sorted((src, dst))            # global lock order -> no deadlock
        with self._locks[first]:
            with self._locks[second]:
                if self._b[src] < amt:
                    raise ValueError("insufficient funds")
                self._b[src] -= amt
                self._b[dst] += amt

    def total(self):
        with ExitStack() as stack:                    # consistent snapshot, same lock order
            for acct in sorted(self._b):
                stack.enter_context(self._locks[acct])
            return sum(self._b.values())


def ref_apply_batch(initial, txns):
    seen = set()
    bal = dict(initial)                               # work on a copy -> free rollback
    for tid, src, dst, amt in txns:
        if tid in seen:
            continue
        if bal[src] < amt:
            raise ValueError("overdraft in batch")
        seen.add(tid)
        bal[src] -= amt
        bal[dst] += amt
    return bal


class RefShardedLedger:
    def __init__(self, n_shards):
        self.n = n_shards
        self.shards = [{} for _ in range(n_shards)]
        self.locks = [threading.Lock() for _ in range(n_shards)]

    def _i(self, acct):
        return hash(acct) % self.n

    def open(self, acct, balance=0):
        i = self._i(acct)
        with self.locks[i]:
            self.shards[i][acct] = balance

    def balance(self, acct):
        i = self._i(acct)
        with self.locks[i]:
            return self.shards[i][acct]

    def transfer(self, src, dst, amt):
        i, j = self._i(src), self._i(dst)
        if i != j:
            raise ValueError("cross-shard transfer not supported")
        with self.locks[i]:
            if self.shards[i][src] < amt:
                raise ValueError("insufficient funds")
            self.shards[i][src] -= amt
            self.shards[i][dst] += amt


def ref_parallel_replay(shards, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return list(ex.map(ledger_workers.replay_shard, shards))


_L = RefLedger(); _L.open("a", 10); _L.open("b", 0); _L.transfer("a", "b", 4)
assert _L.balance("a") == 6 and _L.balance("b") == 4
assert ref_replay_txns({"a": 5, "b": 0}, [("a", "b", 5)]) == {"a": 0, "b": 5}
assert ref_apply_batch({"a": 10, "b": 0}, [("t", "a", "b", 3), ("t", "a", "b", 3)]) == {"a": 7, "b": 3}
assert ref_parallel_replay([({0: 5, 4: 0}, [(0, 4, 9)])], 2) == [{0: 5, 4: 0}]   # overdraft skipped
print("reference solutions OK")